# Análise de desempenho: Serial vs. Paralelo

Usar o arquivo **`comparacao_geral_maior.csv`** para gerar:

- tabela consolidada com tempos;
- tabela pivô por tamanho e método;
- gráfico de tempo de execução;
- gráfico de speedup;
- gráfico de eficiência.

In [3]:
pip install pandas matplotlib

   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ----------- ---------------------------- 2.4/8.1 MB 14.3 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 23.4 MB/s  0:00:00
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 2.3/2.3 MB 70.6 MB/s  0:00:00
   ---------------------------------------- 0.0/7.1 MB ? eta -:--:--
   ----------------------------- ---------- 5.2/7.1 MB 32.9 MB/s eta 0:00:01
   ---------------------------------------- 7.1/7.1 MB 19.7 MB/s  0:00:00

   ---------------------------------------- 0/7 [pyparsing]
   ---------------------------------------- 0/7 [pyparsing]
   ----- ---------------------------------- 1/7 [pillow]
   ----- ---------------------------------- 1/7 [pillow]
   ----- ---------------------------------- 1/7 [pillow]
   ----- ---------------------------------- 1/7 [pillow]
   ----- ---------------------------------- 1/7 [pillow]
   ----- --


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

candidatos = [
    Path("comparacao_serial_parallel/comparacao_geral_maior.csv"),
    Path("comparacao_serial_parallel/comparacao_geral.csv"),
    Path("comparacao_geral_maior.csv"),
    Path("comparacao_geral.csv"),
]

arquivo = None
for caminho in candidatos:
    if caminho.exists():
        arquivo = caminho
        break

if arquivo is None:
    raise FileNotFoundError("Nenhum arquivo CSV de comparação foi encontrado.")

df = pd.read_csv(arquivo)
print(f"Arquivo carregado: {arquivo}")
df.head()

FileNotFoundError: Nenhum arquivo CSV de comparação foi encontrado.

## Tabela consolidada

In [ ]:
df.sort_values(["n", "m", "p", "metodo"]).reset_index(drop=True)

## Tabela pivô por tamanho e método

In [ ]:
tabela_pivot = df.pivot(index="tamanho", columns="metodo", values="tempo_segundos")
tabela_pivot

## Gráfico de tempo de execução

In [ ]:
plt.figure(figsize=(10, 5))

for metodo in df["metodo"].unique():
    base = df[df["metodo"] == metodo]
    plt.plot(base["tamanho"], base["tempo_segundos"], marker="o", label=metodo)

plt.title("Tempo de execução por tamanho e método")
plt.xlabel("Tamanho da matriz")
plt.ylabel("Tempo de execução (s)")
plt.legend()
plt.grid(True)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Gráfico de speedup

In [ ]:
df_speedup = df[df["metodo"] != "Serial"].copy()

pivot_speedup = df_speedup.pivot(index="tamanho", columns="metodo", values="speedup")
pivot_speedup.plot(kind="bar", figsize=(10, 5))
plt.title("Speedup por tamanho e método")
plt.xlabel("Tamanho da matriz")
plt.ylabel("Speedup")
plt.xticks(rotation=20)
plt.grid(axis="y")
plt.tight_layout()
plt.show()

## Gráfico de eficiência

In [ ]:
pivot_eficiencia = df_speedup.pivot(index="tamanho", columns="metodo", values="eficiencia")
pivot_eficiencia.plot(kind="bar", figsize=(10, 5))
plt.title("Eficiência por tamanho e método")
plt.xlabel("Tamanho da matriz")
plt.ylabel("Eficiência")
plt.xticks(rotation=20)
plt.grid(axis="y")
plt.tight_layout()
plt.show()

## Melhor método por tamanho

In [ ]:
somente_paralelos = df[df["metodo"] != "Serial"].copy()
melhor = somente_paralelos.loc[somente_paralelos.groupby("tamanho")["tempo_segundos"].idxmin()]
melhor[["tamanho", "metodo", "tempo_segundos", "speedup", "eficiencia"]]